# 📖 Notebook 3: Search Ranking and Relevance

Finding posts that *match* a query is only half the problem. The other half is **ranking** — showing the *best* results first. When you search "Taylor Swift" on Facebook, you don't want a random post from 5 years ago with 0 likes. You want the most relevant, popular, and recent results.

In this notebook, we'll explore how search engines score and rank results, and how to combine relevance with business signals like recency and popularity.

## Learning Objectives

By the end of this notebook, you'll understand:
- How **TF-IDF** and **BM25** score relevance
- How to sort by **recency** or **like count** in Elasticsearch
- How **function_score** queries combine relevance with business signals
- How **multi-keyword** and **phrase queries** work
- The **two-stage architecture** pattern used in production search systems

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd system-designs/fb-post-search
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
from elasticsearch import Elasticsearch, helpers
import time
import json
import pandas as pd

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "fb_post_search",
    "user": "demo",
    "password": "demo"
}

es = Elasticsearch("http://localhost:9200")

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Verify connections
try:
    conn = get_db()
    conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    es.info()
    print("✅ Elasticsearch connected")
except Exception as e:
    print(f"❌ Elasticsearch: {e}")

✅ PostgreSQL connected
✅ Elasticsearch connected


In [2]:
# Ensure the 'posts' index exists (created in Notebook 1)
# If not, re-create and populate it

INDEX_NAME = "posts"

if not es.indices.exists(index=INDEX_NAME):
    print("Index 'posts' not found — creating it now...")
    es.indices.create(
        index=INDEX_NAME,
        body={
            "settings": {
                "number_of_shards": 1,
                "number_of_replicas": 0,
                "analysis": {
                    "analyzer": {
                        "post_analyzer": {
                            "type": "custom",
                            "tokenizer": "standard",
                            "filter": ["lowercase", "stop", "snowball"]
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "content":    {"type": "text", "analyzer": "post_analyzer"},
                    "user_id":    {"type": "integer"},
                    "like_count": {"type": "integer"},
                    "created_at": {"type": "date"}
                }
            }
        }
    )

    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT id, user_id, content, like_count, created_at FROM posts")
    rows = cur.fetchall()
    conn.close()

    actions = [{
        "_index": INDEX_NAME, "_id": r[0],
        "_source": {"user_id": r[1], "content": r[2], "like_count": r[3],
                    "created_at": r[4].isoformat() if r[4] else None}
    } for r in rows]
    helpers.bulk(es, actions)
    es.indices.refresh(index=INDEX_NAME)
    print(f"✅ Indexed {len(actions)} posts")
else:
    count = es.count(index=INDEX_NAME)["count"]
    print(f"✅ Index 'posts' exists with {count} documents")

✅ Index 'posts' exists with 508 documents


## 📐 How Search Scoring Works: TF-IDF and BM25

When you search for "coffee", Elasticsearch gives each matching post a **score**. Higher scores = shown first. But how does it decide which post is more relevant?

### TF-IDF (the foundation)

**TF-IDF** stands for **Term Frequency × Inverse Document Frequency**:

- **Term Frequency (TF)**: How often does "coffee" appear in *this* post?  
  → A post mentioning "coffee" 5 times is probably more about coffee than one mentioning it once.

- **Inverse Document Frequency (IDF)**: How rare is "coffee" across *all* posts?  
  → If "coffee" appears in 5 out of 500 posts, it's a meaningful search term.  
  → If "the" appears in 495 out of 500 posts, it's useless for ranking.

```
TF-IDF score = TF × IDF
             = (times word appears in doc) × log(total docs / docs with word)
```

### BM25 (the upgrade)

**BM25** (Best Matching 25) is TF-IDF with two improvements:

1. **Saturation**: After a word appears many times, the score increase slows down. Mentioning "coffee" 50 times isn't 50× more relevant than mentioning it once.

2. **Document length normalization**: A short post that mentions "coffee" once is probably more focused on coffee than a 1000-word essay that mentions it once.

**Elasticsearch uses BM25 by default.** You rarely need to change this.

In [3]:
# Let's see BM25 in action — search for "coffee" and examine scores

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {"match": {"content": "coffee"}},
        "explain": True,
        "size": 5
    }
)

print("☕ BM25 Scores for 'coffee' search")
print("=" * 70)

for hit in result["hits"]["hits"]:
    score = hit["_score"]
    content = hit["_source"]["content"][:70]
    likes = hit["_source"]["like_count"]

    # Extract BM25 components from explanation
    explanation = hit["_explanation"]
    print(f"\n  Score: {score:.4f} | {likes} likes")
    print(f"  Content: {content}...")
    print(f"  Scoring: {explanation['description'][:80]}")

print()
print("💡 Posts that mention 'coffee' more often and are shorter get higher BM25 scores.")
print("   But notice: BM25 ignores likes and recency entirely!")

☕ BM25 Scores for 'coffee' search

  Score: 4.1115 | 678 likes
  Content: Coffee is fuel for programmers. My morning ritual: brew coffee, open t...
  Scoring: weight(content:coffe in 47) [PerFieldSimilarity], result of:

  Score: 4.0231 | 345 likes
  Content: The best coffee shop in San Francisco serves the most amazing pour-ove...
  Scoring: weight(content:coffe in 46) [PerFieldSimilarity], result of:

  Score: 4.0231 | 1234 likes
  Content: Tried quitting coffee for a month. Worst month of my life. Coffee and ...
  Scoring: weight(content:coffe in 48) [PerFieldSimilarity], result of:

  Score: 3.0240 | 456 likes
  Content: The science behind coffee: caffeine blocks adenosine receptors, which ...
  Scoring: weight(content:coffe in 49) [PerFieldSimilarity], result of:

  Score: 2.9294 | 890 likes
  Content: Cold brew coffee recipe: coarse grounds, cold water, 12 hours in the f...
  Scoring: weight(content:coffe in 50) [PerFieldSimilarity], result of:

💡 Posts that mention 'coffee' more

## 📅 Sorting by Recency

The Facebook Post Search design requires sorting by **recency** (newest first) or **like count** (most popular first). BM25 only measures text relevance — we need to add our own sorting.

The simplest approach: ignore BM25 scoring and sort by a field.

In [4]:
# Sort by recency (newest posts first)

def search_by_recency(keyword, size=10):
    """Find matching posts, sorted by newest first."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {"match": {"content": keyword}},
            "sort": [{"created_at": {"order": "desc"}}],
            "size": size
        }
    )

result = search_by_recency("coffee")

print("📅 Search 'coffee' — sorted by RECENCY (newest first)")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  [{date}] {src['like_count']:>5} likes | {src['content'][:55]}...")

📅 Search 'coffee' — sorted by RECENCY (newest first)
  [2026-04-11]  1425 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-03-28]  4090 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-03-25]  2340 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-03-10]  1590 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-03-07]  2510 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-03-06]   935 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-03-04]   345 likes | The best coffee shop in San Francisco serves the most a...
  [2026-03-03]   678 likes | Coffee is fuel for programmers. My morning ritual: brew...
  [2026-03-02]  1234 likes | Tried quitting coffee for a month. Worst month of my li...
  [2026-03-01]   456 likes | The science behind coffee: caffeine blocks adenosine re...


In [5]:
# Sort by like count (most popular first)

def search_by_likes(keyword, size=10):
    """Find matching posts, sorted by most liked."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {"match": {"content": keyword}},
            "sort": [{"like_count": {"order": "desc"}}],
            "size": size
        }
    )

result = search_by_likes("coffee")

print("👍 Search 'coffee' — sorted by LIKES (most popular first)")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  [{date}] {src['like_count']:>5} likes | {src['content'][:55]}...")

👍 Search 'coffee' — sorted by LIKES (most popular first)
  [2026-02-17]  4926 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-01-26]  4572 likes | New coffee shop opened downtown. The espresso is strong...
  [2025-11-04]  4512 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-03-28]  4090 likes | New coffee shop opened downtown. The espresso is strong...
  [2025-11-11]  3569 likes | New coffee shop opened downtown. The espresso is strong...
  [2025-12-14]  3082 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-02-02]  2919 likes | New coffee shop opened downtown. The espresso is strong...
  [2025-12-31]  2910 likes | New coffee shop opened downtown. The espresso is strong...
  [2025-10-25]  2779 likes | New coffee shop opened downtown. The espresso is strong...
  [2026-03-07]  2510 likes | New coffee shop opened downtown. The espresso is strong...


## 🎯 Combining Relevance + Popularity + Recency

In the real world, you don't want *just* relevance, *just* recency, or *just* likes. You want a **blend**:

- A post from 10 minutes ago with 5 likes about "coffee" should beat
- A post from 2 years ago with 100 likes that barely mentions "coffee"

Elasticsearch's **`function_score`** query lets you combine BM25 with custom scoring functions. Think of it as:

```
final_score = BM25_score × f(like_count) × f(recency)
```

This is the heart of how Facebook and other platforms rank search results.

In [6]:
# function_score: Boost by like count
# Uses log1p(like_count) so popular posts score higher,
# but the boost saturates (1000 likes isn't 1000× better than 1 like)

def search_boosted_by_likes(keyword, size=10):
    """BM25 relevance boosted by popularity (like count)."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "function_score": {
                    "query": {"match": {"content": keyword}},
                    "functions": [
                        {
                            "field_value_factor": {
                                "field": "like_count",
                                "modifier": "log1p",    # log(1 + like_count)
                                "factor": 2             # weight of this signal
                            }
                        }
                    ],
                    "boost_mode": "multiply"   # final = BM25 × like_boost
                }
            },
            "size": size
        }
    )

result = search_boosted_by_likes("coffee")

print("🎯 Search 'coffee' — BM25 × Popularity Boost")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  score={hit['_score']:>8.2f} | [{date}] {src['like_count']:>5} likes | {src['content'][:45]}...")

print()
print("💡 Now popular posts rank higher, but relevance still matters.")
print("   A very relevant post with few likes can still beat a mildly")
print("   relevant post with many likes.")

🎯 Search 'coffee' — BM25 × Popularity Boost
  score=   13.65 | [2026-03-02]  1234 likes | Tried quitting coffee for a month. Worst mont...
  score=   12.88 | [2026-03-03]   678 likes | Coffee is fuel for programmers. My morning ri...
  score=   11.70 | [2026-02-17]  4926 likes | New coffee shop opened downtown. The espresso...
  score=   11.60 | [2026-01-26]  4572 likes | New coffee shop opened downtown. The espresso...
  score=   11.59 | [2025-11-04]  4512 likes | New coffee shop opened downtown. The espresso...
  score=   11.46 | [2026-03-28]  4090 likes | New coffee shop opened downtown. The espresso...
  score=   11.42 | [2026-03-04]   345 likes | The best coffee shop in San Francisco serves ...
  score=   11.29 | [2025-11-11]  3569 likes | New coffee shop opened downtown. The espresso...
  score=   11.10 | [2025-12-14]  3082 likes | New coffee shop opened downtown. The espresso...
  score=   11.03 | [2026-02-02]  2919 likes | New coffee shop opened downtown. The espresso...

💡 Now

In [7]:
# function_score: Boost by recency using exponential decay
# Posts lose score as they age. Recent posts get a big boost.

def search_boosted_by_recency(keyword, size=10):
    """BM25 relevance boosted by recency (newer = higher score)."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "function_score": {
                    "query": {"match": {"content": keyword}},
                    "functions": [
                        {
                            "exp": {
                                "created_at": {
                                    "origin": "now",     # center = right now
                                    "scale": "7d",       # half-life = 7 days
                                    "decay": 0.5         # score halves every 7 days
                                }
                            }
                        }
                    ],
                    "boost_mode": "multiply"
                }
            },
            "size": size
        }
    )

result = search_boosted_by_recency("coffee")

print("📅 Search 'coffee' — BM25 × Recency Decay")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  score={hit['_score']:>8.4f} | [{date}] {src['like_count']:>5} likes | {src['content'][:45]}...")

print()
print("💡 Exponential decay means:")
print("   - Posts from today: ~100% of BM25 score")
print("   - Posts from 7 days ago: ~50% of BM25 score")
print("   - Posts from 14 days ago: ~25% of BM25 score")
print("   - Posts from 30 days ago: ~6% of BM25 score")

📅 Search 'coffee' — BM25 × Recency Decay
  score=  1.3301 | [2026-04-11]  1425 likes | New coffee shop opened downtown. The espresso...
  score=  0.3096 | [2026-03-28]  4090 likes | New coffee shop opened downtown. The espresso...
  score=  0.2288 | [2026-03-25]  2340 likes | New coffee shop opened downtown. The espresso...
  score=  0.0517 | [2026-03-10]  1590 likes | New coffee shop opened downtown. The espresso...
  score=  0.0423 | [2026-03-04]   345 likes | The best coffee shop in San Francisco serves ...
  score=  0.0391 | [2026-03-03]   678 likes | Coffee is fuel for programmers. My morning ri...
  score=  0.0391 | [2026-03-07]  2510 likes | New coffee shop opened downtown. The espresso...
  score=  0.0364 | [2026-03-06]   935 likes | New coffee shop opened downtown. The espresso...
  score=  0.0347 | [2026-03-02]  1234 likes | Tried quitting coffee for a month. Worst mont...
  score=  0.0236 | [2026-03-01]   456 likes | The science behind coffee: caffeine blocks ad...

💡 Expone

In [8]:
# The full combo: BM25 × popularity × recency

def search_combined_ranking(keyword, size=10):
    """BM25 relevance boosted by both popularity AND recency."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "function_score": {
                    "query": {"match": {"content": keyword}},
                    "functions": [
                        {
                            "field_value_factor": {
                                "field": "like_count",
                                "modifier": "log1p",
                                "factor": 1.5
                            }
                        },
                        {
                            "exp": {
                                "created_at": {
                                    "origin": "now",
                                    "scale": "14d",
                                    "decay": 0.5
                                }
                            }
                        }
                    ],
                    "score_mode": "multiply",   # combine functions by multiplying
                    "boost_mode": "multiply"     # combine with BM25 by multiplying
                }
            },
            "size": size
        }
    )

result = search_combined_ranking("coffee")

print("🏆 Search 'coffee' — BM25 × Popularity × Recency")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  score={hit['_score']:>8.2f} | [{date}] {src['like_count']:>5} likes | {src['content'][:45]}...")

print()
print("💡 This is the 'production recipe': BM25 × popularity × recency.")
print("   Tune the weights (factor, scale, decay) based on your product needs.")

🏆 Search 'coffee' — BM25 × Popularity × Recency
  score=    6.57 | [2026-04-11]  1425 likes | New coffee shop opened downtown. The espresso...
  score=    3.61 | [2026-03-28]  4090 likes | New coffee shop opened downtown. The espresso...
  score=    2.90 | [2026-03-25]  2340 likes | New coffee shop opened downtown. The espresso...
  score=    1.31 | [2026-03-10]  1590 likes | New coffee shop opened downtown. The espresso...
  score=    1.22 | [2026-03-02]  1234 likes | Tried quitting coffee for a month. Worst mont...
  score=    1.21 | [2026-03-07]  2510 likes | New coffee shop opened downtown. The espresso...
  score=    1.21 | [2026-03-03]   678 likes | Coffee is fuel for programmers. My morning ri...
  score=    1.12 | [2026-03-04]   345 likes | The best coffee shop in San Francisco serves ...
  score=    1.03 | [2026-03-06]   935 likes | New coffee shop opened downtown. The espresso...
  score=    0.77 | [2026-02-28]   890 likes | Cold brew coffee recipe: coarse grounds, cold...

💡

## 🔗 Multi-Keyword and Phrase Queries

Users don't always search for single words. What happens when someone searches for "Taylor Swift"?

There are three ways to handle this:

1. **OR match**: Posts containing "Taylor" OR "Swift" (most results, lowest precision)
2. **AND match**: Posts containing "Taylor" AND "Swift" (fewer results, higher precision)
3. **Phrase match**: Posts containing the exact phrase "Taylor Swift" (fewest results, highest precision)

In [9]:
# Compare: OR vs AND vs Phrase matching for "Taylor Swift"

# OR match (default) — either word matches
or_result = es.search(
    index=INDEX_NAME,
    body={"query": {"match": {"content": "Taylor Swift"}}, "size": 10}
)

# AND match — both words must appear (but not necessarily together)
and_result = es.search(
    index=INDEX_NAME,
    body={"query": {"match": {"content": {"query": "Taylor Swift", "operator": "and"}}}, "size": 10}
)

# Phrase match — words must appear together in order
phrase_result = es.search(
    index=INDEX_NAME,
    body={"query": {"match_phrase": {"content": "Taylor Swift"}}, "size": 10}
)

print("🔗 Multi-Keyword Query: 'Taylor Swift'")
print("=" * 70)
print(f"  OR match:     {or_result['hits']['total']['value']} results (Taylor OR Swift)")
print(f"  AND match:    {and_result['hits']['total']['value']} results (Taylor AND Swift)")
print(f"  Phrase match: {phrase_result['hits']['total']['value']} results (exact phrase)")

print("\n--- OR match results (notice some are about 'swift' but not Taylor Swift) ---")
for hit in or_result["hits"]["hits"][:3]:
    print(f"  [score={hit['_score']:.2f}] {hit['_source']['content'][:65]}...")

print("\n--- Phrase match results (exact 'Taylor Swift') ---")
for hit in phrase_result["hits"]["hits"][:3]:
    print(f"  [score={hit['_score']:.2f}] {hit['_source']['content'][:65]}...")

🔗 Multi-Keyword Query: 'Taylor Swift'
  OR match:     4 results (Taylor OR Swift)
  AND match:    4 results (Taylor AND Swift)
  Phrase match: 2 results (exact phrase)

--- OR match results (notice some are about 'swift' but not Taylor Swift) ---
  [score=10.13] Just saw Taylor Swift in concert. The production quality and ener...
  [score=9.80] My cat is named Taylor and she is swift at catching mice. Best na...
  [score=9.20] Taylor Swift Eras Tour broke every concert record. The setlist sp...

--- Phrase match results (exact 'Taylor Swift') ---
  [score=10.13] Just saw Taylor Swift in concert. The production quality and ener...
  [score=9.20] Taylor Swift Eras Tour broke every concert record. The setlist sp...


In [10]:
# Best practice: use bool query to combine phrase + OR for best results
# Phrase matches get a big boost, but OR matches still appear

def search_smart_multi_keyword(query_text, size=10):
    """Smart multi-keyword search: boost exact phrases, include partial matches."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "bool": {
                    "must": [
                        {"match": {"content": query_text}}
                    ],
                    "should": [
                        {"match_phrase": {"content": {"query": query_text, "boost": 3}}}
                    ]
                }
            },
            "size": size
        }
    )

result = search_smart_multi_keyword("Taylor Swift")

print("🧠 Smart Search: 'Taylor Swift' (phrase boosted 3×)")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    print(f"  score={hit['_score']:>6.2f} | {src['like_count']:>5} likes | {src['content'][:55]}...")

print()
print("💡 Posts with the exact phrase 'Taylor Swift' rank highest,")
print("   but posts mentioning either word still appear.")

🧠 Smart Search: 'Taylor Swift' (phrase boosted 3×)
  score= 40.51 |  4567 likes | Just saw Taylor Swift in concert. The production qualit...
  score= 36.82 |  8901 likes | Taylor Swift Eras Tour broke every concert record. The ...
  score=  9.80 |   234 likes | My cat is named Taylor and she is swift at catching mic...
  score=  9.20 |   123 likes | Taylor made a swift decision to learn programming. Thre...

💡 Posts with the exact phrase 'Taylor Swift' rank highest,
   but posts mentioning either word still appear.


## 🏗️ Two-Stage Architecture

In the FB Post Search design, the like counts in the index may be **stale** (to reduce write volume, likes are only updated at milestones like powers of 2). This leads to a **two-stage architecture**:

```
Stage 1: RETRIEVE (fast, approximate)           Stage 2: RE-RANK (precise)
┌─────────────────────────────────┐     ┌────────────────────────────────────┐
│  Elasticsearch                  │     │  Search Service                    │
│                                 │     │                                    │
│  Query: "coffee"                │     │  For each of 20 candidates:        │
│  Get top 2N results using       │────→│  1. Fetch real-time like count     │
│  approximate like counts        │     │  2. Re-sort by actual like count   │
│                                 │     │  3. Return top N to user           │
└─────────────────────────────────┘     └────────────────────────────────────┘
```

**Why 2N?** We fetch more than we need because the approximate ranking might be slightly wrong. After re-ranking with fresh data, we take the top N.

This pattern is extremely common in information retrieval and recommendation systems. Let's implement it.

In [11]:
# Simulate the two-stage architecture

def two_stage_search(keyword, n=5):
    """
    Two-stage search:
    Stage 1: Get 2N candidates from Elasticsearch (approximate ranking)
    Stage 2: Re-rank with real-time like counts from PostgreSQL
    """
    # --- Stage 1: Retrieve candidates from Elasticsearch ---
    es_result = es.search(
        index=INDEX_NAME,
        body={
            "query": {"match": {"content": keyword}},
            "sort": [{"like_count": {"order": "desc"}}],
            "size": n * 2   # fetch 2× what we need
        }
    )

    candidates = []
    for hit in es_result["hits"]["hits"]:
        candidates.append({
            "post_id": int(hit["_id"]),
            "content": hit["_source"]["content"],
            "es_likes": hit["_source"]["like_count"]   # possibly stale
        })

    # --- Stage 2: Fetch real-time like counts from PostgreSQL ---
    post_ids = [c["post_id"] for c in candidates]
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT id, like_count FROM posts WHERE id = ANY(%s)",
        (post_ids,)
    )
    real_likes = dict(cur.fetchall())
    conn.close()

    # Attach real-time like counts
    for c in candidates:
        c["real_likes"] = real_likes.get(c["post_id"], c["es_likes"])

    # Re-sort by real-time like count
    candidates.sort(key=lambda x: -x["real_likes"])

    return candidates[:n]  # return top N

# Run the two-stage search
start = time.time()
results = two_stage_search("coffee", n=5)
elapsed = (time.time() - start) * 1000

print(f"🏗️ Two-Stage Search for 'coffee' ({elapsed:.1f}ms total)")
print("=" * 70)
print(f"  {'Post ID':<10} {'ES Likes (approx)':<20} {'Real Likes':<15} {'Content'}")
print("-" * 70)
for c in results:
    print(f"  {c['post_id']:<10} {c['es_likes']:<20} {c['real_likes']:<15} {c['content'][:40]}...")

print()
print("💡 In production, ES likes might differ from real likes because")
print("   we only update the index at milestones (1, 2, 4, 8, 16...).")
print("   Stage 2 fetches the true count for precise ranking.")

🏗️ Two-Stage Search for 'coffee' (14.2ms total)
  Post ID    ES Likes (approx)    Real Likes      Content
----------------------------------------------------------------------
  329        4926                 4926            New coffee shop opened downtown. The esp...
  309        4572                 4572            New coffee shop opened downtown. The esp...
  109        4512                 4512            New coffee shop opened downtown. The esp...
  189        4090                 4090            New coffee shop opened downtown. The esp...
  469        3569                 3569            New coffee shop opened downtown. The esp...

💡 In production, ES likes might differ from real likes because
   we only update the index at milestones (1, 2, 4, 8, 16...).
   Stage 2 fetches the true count for precise ranking.


## 📊 Ranking Comparison Dashboard

Let's compare all ranking strategies side by side for the same query.

In [12]:
# Compare ranking strategies for "python"

keyword = "python"

strategies = {
    "BM25 Only": es.search(index=INDEX_NAME, body={
        "query": {"match": {"content": keyword}}, "size": 5
    }),
    "By Recency": search_by_recency(keyword, 5),
    "By Likes": search_by_likes(keyword, 5),
    "Combined": search_combined_ranking(keyword, 5),
}

for name, result in strategies.items():
    print(f"\n{'=' * 70}")
    print(f"📊 Strategy: {name}")
    print(f"{'=' * 70}")
    for i, hit in enumerate(result["hits"]["hits"], 1):
        src = hit["_source"]
        date = src["created_at"][:10] if src["created_at"] else "unknown"
        # When sorting by a field, Elasticsearch returns _score=None, so guard against it.
        score_str = f"{hit['_score']:>8.2f}" if hit.get('_score') is not None else "    n/a "
        print(f"  #{i} | score={score_str} | {date} | {src['like_count']:>5} likes | {src['content'][:40]}...")

print("\n💡 Notice how different strategies surface different posts!")
print("   The 'right' ranking depends on what your users want.")


📊 Strategy: BM25 Only
  #1 | score=    3.29 | 2026-03-07 |  1787 likes | Working on a new project with Python and...
  #2 | score=    3.29 | 2025-12-10 |  3256 likes | Working on a new project with Python and...
  #3 | score=    3.29 | 2026-01-21 |  2230 likes | Working on a new project with Python and...
  #4 | score=    3.29 | 2025-11-27 |   507 likes | Working on a new project with Python and...
  #5 | score=    3.29 | 2026-02-19 |   102 likes | Working on a new project with Python and...

📊 Strategy: By Recency
  #1 | score=    n/a  | 2026-04-19 |   234 likes | Just finished building a REST API with P...
  #2 | score=    n/a  | 2026-04-17 |   776 likes | Working on a new project with Python and...
  #3 | score=    n/a  | 2026-04-15 |  2200 likes | Working on a new project with Python and...
  #4 | score=    n/a  | 2026-04-12 |  4198 likes | Working on a new project with Python and...
  #5 | score=    n/a  | 2026-04-06 |  2325 likes | Working on a new project with Python and...

📊 

## 🧹 Cleanup

In [13]:
# Clean up all Elasticsearch indices

for idx in ["posts", "posts_autocomplete", "search_suggestions"]:
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)
        print(f"🧹 Deleted index: {idx}")

print()
print("To stop all containers:")
print("  cd system-designs/fb-post-search")
print("  docker-compose down -v")

🧹 Deleted index: posts
🧹 Deleted index: posts_autocomplete
🧹 Deleted index: search_suggestions

To stop all containers:
  cd system-designs/fb-post-search
  docker-compose down -v


## 📚 Summary

### Key Takeaways

1. **BM25 measures text relevance** — it considers term frequency, document frequency, and document length. Elasticsearch uses it by default.
2. **Sorting by a field** (recency or likes) ignores relevance entirely. Simple but sometimes useful.
3. **`function_score` combines signals** — multiply BM25 with popularity boosts (log1p) and recency decay (exponential). This is the production recipe.
4. **Multi-keyword queries** have three modes: OR (broadest), AND (stricter), and phrase match (strictest). Use `bool` queries to boost exact phrases.
5. **Two-stage architecture** — use approximate scores for fast retrieval (Stage 1), then re-rank with precise data (Stage 2). This is how production systems handle stale indexes.

### How This Maps to the Facebook Post Search Design

| Concept | How It's Used |
|---------|---------------|
| Inverted Index | Maps keywords → post IDs in Redis/Elasticsearch |
| BM25 Scoring | Ranks posts by text relevance |
| Recency Sorting | Creation index sorted by timestamp |
| Like Count Sorting | Likes index using Redis sorted sets |
| Two-Stage Architecture | Approximate ranking → fetch real likes → re-sort |
| Phrase Matching | Bigram indexes or set intersection |
| Caching | CDN + Redis cache for repeated queries |

### 🎓 You've Completed the FB Post Search Lab!

You now understand the core search concepts that come up in system design interviews:
- **Notebook 1**: How inverted indexes make search fast
- **Notebook 2**: How typeahead and autocomplete work
- **Notebook 3**: How to rank and combine multiple signals

### Further Reading

- [Elasticsearch: The Definitive Guide](https://www.elastic.co/guide/en/elasticsearch/reference/current/index.html)
- [Hello Interview: FB Post Search Design](https://www.hellointerview.com/learn/system-design/problem-breakdowns/fb-post-search)
- [BM25 Algorithm Explained](https://en.wikipedia.org/wiki/Okapi_BM25)